In [1]:
!wget -O /home/jovyan/work/data/yellow_tripdata_2025-11.parquet \
    https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

!wget -O /home/jovyan/work/data/taxi_zone_lookup.csv \
    https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-13 19:20:42--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.171.57.179, 3.171.57.190, 3.171.57.103, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.171.57.179|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘/home/jovyan/work/data/yellow_tripdata_2025-11.parquet’

/home/jovyan/work/d 100%[===================>]  67.84M  31.2MB/s    in 2.2s    

2026-03-13 19:20:44 (31.2 MB/s) - ‘/home/jovyan/work/data/yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]

--2026-03-13 19:20:44--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 3.171.57.179, 3.171.57.190, 3.171.57.103, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|3.171.57.179|:443...

In [3]:
# Question 1
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Homework6") \
    .getOrCreate()

spark.version


'3.5.0'

In [5]:
# Question 2
df = spark.read.parquet("/home/jovyan/work/data/yellow_tripdata_2025-11.parquet")
df.repartition(4).write.parquet("/home/jovyan/work/data/output/yellow_2025_11/", mode="overwrite")

In [6]:
# Question 3
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [7]:
df.createOrReplaceTempView("yellow_taxi")

spark.sql("""
    SELECT COUNT(*)
    FROM yellow_taxi
    WHERE CAST(tpep_pickup_datetime AS DATE) = '2025-11-15'
""").show()


+--------+
|count(1)|
+--------+
|  162604|
+--------+



In [8]:
# Question 4
spark.sql("""
    SELECT MAX(
        TIMESTAMPDIFF(SECOND, tpep_pickup_datetime, tpep_dropoff_datetime) / 3600
    ) AS max_duration_hours
    FROM yellow_taxi
""").show()


+------------------+
|max_duration_hours|
+------------------+
| 90.64666666666666|
+------------------+



In [9]:
# Question 6
zones = spark.read.csv(
    "/home/jovyan/work/data/taxi_zone_lookup.csv", 
    header=True, 
    inferSchema=True
)
zones.createOrReplaceTempView("zones")


In [10]:
zones.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows



In [11]:
spark.sql("""
    SELECT z.Zone, COUNT(*) AS pickup_count
    FROM yellow_taxi t
    JOIN zones z ON t.PULocationID = z.LocationID
    GROUP BY z.Zone
    ORDER BY pickup_count ASC
    LIMIT 5
""").show(truncate=False)

+---------------------------------------------+------------+
|Zone                                         |pickup_count|
+---------------------------------------------+------------+
|Governor's Island/Ellis Island/Liberty Island|1           |
|Eltingville/Annadale/Prince's Bay            |1           |
|Arden Heights                                |1           |
|Port Richmond                                |3           |
|Rikers Island                                |4           |
+---------------------------------------------+------------+

